# **Research Objective**
This study aims to identify the key determinants of apartment prices and evaluate how structural characteristics, building features, and market timing affect housing prices.

We want to check whether new buildings, renovation, or floor level significantly affect prices

In [ ]:
import re
import pandas as pd

In [ ]:
# Load data
data = pd.read_csv("/content/scraped_data.csv")
# data.head()

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23483 entries, 0 to 23482
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Price         23433 non-null  object 
 1   Address       23483 non-null  object 
 2   Title         23483 non-null  object 
 3   Fast          0 non-null      float64
 4   Broker        0 non-null      float64
 5   Type          23483 non-null  object 
 6   New_building  23483 non-null  object 
 7   nFloor        23483 non-null  object 
 8   Floor         23483 non-null  object 
 9   Height        23483 non-null  object 
 10  Sqm           23483 non-null  object 
 11  Rooms         23483 non-null  object 
 12  Bathrooms     23483 non-null  object 
 13  Renovation    23483 non-null  object 
 14  Balcony       23483 non-null  object 
 15  Body          23481 non-null  object 
 16  ID            23483 non-null  int64  
 17  P_Date        23483 non-null  object 
 18  R_Date        22628 non-nu

In [ ]:
def converter(price):
    currency = re.search(r"\$|€|֏|руб|₽|nan", price)[0]
    return price.replace(currency, "").replace(",", ""), currency

# Clean `Price` column
data['Price'].dropna(inplace=True)
data['Price'] = data['Price'].astype(str)

# Separate currency and convert `Price` to numeric
data['Price'], data["Currency"] = zip(*data.Price.apply(converter))
data['Price'] = pd.to_numeric(data['Price'], errors='coerce')
print(f"{data['Price'].count()} valid entries.")

23433 valid entries.


In [ ]:
data = data[data['Currency']=='$']

We need to drop the duplicates. This

* Keeps economic information

* Reduces noise from negotiation, timing, or listing behavior

* Prevents biased regression results

In [ ]:
# Drop unnecessary columns
columns_to_drop = ['Fast', 'Broker', 'Title', 'Body', 'Seller_name', 'R_Date', 'P_Date', 'ID', 'HTML_ID']
data.drop(columns=columns_to_drop, axis=1, inplace=True, errors='ignore')
print(f"Remaining columns: {data.columns.tolist()}.")

Remaining columns: ['Price', 'Address', 'Type', 'New_building', 'nFloor', 'Floor', 'Height', 'Sqm', 'Rooms', 'Bathrooms', 'Renovation', 'Balcony', 'Currency'].


In [ ]:
# Remove duplicates
initial_duplicate_count = data.shape[0]
data = data.drop_duplicates(keep='last')
print(f"Removed {initial_duplicate_count - data.shape[0]} duplicates, {data.shape[0]} records remaining.")


Removed 908 duplicates, 21810 records remaining.


In [ ]:
data.head()

,Price,Address,Type,New_building,nFloor,Floor,Height,Sqm,Rooms,Bathrooms,Renovation,Balcony,Currency
0,95000.0,"Օհանովի փողոց, Երևան",Պանելային,Ոչ,9,6,"2,8 մ",68 քմ,2,1,Կապիտալ վերանորոգված,Փակ պատշգամբ,$
1,300000.0,Պուշկինի փողոց,Քարե,Ոչ,9,1,"2,8 մ",58 քմ,2,1,Դիզայներական ոճով վերանորոգված,Առկա չէ,$
2,87000.0,"Նորաշեն թաղամաս, Երևան",Մոնոլիտ,Այո,12,6,3 մ,46 քմ,2,1,Կապիտալ վերանորոգված,Բաց պատշգամբ,$
3,130000.0,"Սուրբ Գրիգոր Լուսավորչի փողոց, Երևան",Քարե,Ոչ,4,2,3 մ,50 քմ,1,1,Մասնակի վերանորոգում,Փակ պատշգամբ,$
4,145000.0,"Գյուլբենկյան փողոց, Երևան",Քարե,Ոչ,5,1,"2,75 մ",70 քմ,3,1,Կապիտալ վերանորոգված,Մի քանի պատշգամբ,$


In [ ]:
feature_cols = [
    'Address', 'New_building',
    'nFloor', 'Floor', 'Height', 'Renovation',
    'Sqm', 'Rooms', 'Bathrooms', 'Balcony', 'Price'
]
data = data[feature_cols]

In [ ]:
# Map districts from geocoded addresses list
address_districts_map = pd.read_csv("/content/all_addresses_with_districts.csv")
data['Address'] = data['Address'].str.strip().str.lower()
data = data.merge(address_districts_map[["Address", "District"]], on='Address', how='left')
data.drop(columns=['Address'], inplace=True)

In [ ]:
# Remove duplicates
initial_na_count = data.shape[0]
data.dropna(inplace=True)
print(f"Removed {initial_na_count - data.shape[0]} NaNs, {data.shape[0]} records remaining.")

Removed 3127 NaNs, 18683 records remaining.


In [ ]:
data.head()

,New_building,nFloor,Floor,Height,Renovation,Sqm,Rooms,Bathrooms,Balcony,Price,District
0,Ոչ,9,6,"2,8 մ",Կապիտալ վերանորոգված,68 քմ,2,1,Փակ պատշգամբ,95000.0,Մալաթիա-Սեբաստիա
2,Այո,12,6,3 մ,Կապիտալ վերանորոգված,46 քմ,2,1,Բաց պատշգամբ,87000.0,Աջափնյակ
3,Ոչ,4,2,3 մ,Մասնակի վերանորոգում,50 քմ,1,1,Փակ պատշգամբ,130000.0,Կենտրոն
4,Ոչ,5,1,"2,75 մ",Կապիտալ վերանորոգված,70 քմ,3,1,Մի քանի պատշգամբ,145000.0,Արաբկիր
5,Այո,8,5,3 մ,Կապիտալ վերանորոգված,47 քմ,2,1,Բաց պատշգամբ,160000.0,Արաբկիր


In [ ]:
# Filtering out invalid 'Sqm' values
initial_sqm_count = data.shape[0]
data['Sqm'] = data['Sqm'].astype(str)
data['Sqm'] = data['Sqm'].apply(lambda x: x.replace("քմ", "").strip())
data['Sqm'] = pd.to_numeric(data['Sqm'], errors='coerce')
print(f"{initial_sqm_count} initial records, {data['Sqm'].count()} valid entries after cleaning.")

18683 initial records, 18683 valid entries after cleaning.


In [ ]:
print(data['Height'].isna().sum())
print(data['Height'].unique())

0
['2,8 մ' '3 մ' '2,75 մ' '3,5 մ' '2,6 մ' '2,7 մ' '3,2 մ' '2,5 մ']


In [ ]:
# Clean and convert 'Height' column
initial_height_count = data['Height'].count()
data['Height'] = data['Height'].astype(str)
data['Height'] = data['Height'].apply(lambda x: x.replace(" մ","").replace(",", ".").strip())
data['Height'] = pd.to_numeric(data['Height'], errors='coerce')
print(f"{initial_height_count} initial records, {data['Height'].count()} valid entries after cleaning.")

18683 initial records, 18683 valid entries after cleaning.


In [ ]:
print(data['Rooms'].unique())
print(data['Bathrooms'].unique())

['2' '1' '3' '4' '5' '6' '7' '8+']
['1' '2' '3+']


In [ ]:
# Cleaning 'Rooms' and 'Bathrooms' columns
initial_rooms_count = data['Rooms'].count()
data['Rooms'] = data['Rooms'].replace("8+", "8")
data['Rooms'] = pd.to_numeric(data['Rooms'], errors='coerce')
print(f"{initial_rooms_count} initial records, {data['Rooms'].count()} valid entries after cleaning `Rooms`.")



initial_bathrooms_count = data['Bathrooms'].count()
data['Bathrooms'] = data['Bathrooms'].replace("3+", "3")
data['Bathrooms'] = pd.to_numeric(data['Bathrooms'], errors='coerce')
print(f"{initial_bathrooms_count} initial records, {data['Bathrooms'].count()} valid entries after cleaning `Bathrooms`.")


18683 initial records, 18683 valid entries after cleaning `Rooms`.
18683 initial records, 18683 valid entries after cleaning `Bathrooms`.


In [ ]:
# Clean 'Floor' and 'nFloor'
initial_floor_count = data['Floor'].count()
# Since, according to our assumption,
# 25 is the maximum number of floor that can appear in Yerevan => anything larger is considered as mistake and is replaced
data['Floor'] = data['Floor'].replace("32+", "25")
data['nFloor'] = data['nFloor'].replace("32+", "25")
data['Floor'] = pd.to_numeric(data['Floor'], errors='coerce')
data['nFloor'] = pd.to_numeric(data['nFloor'], errors='coerce')
print(f"{initial_floor_count} initial records, {data['Floor'].count()} valid entries after cleaning.")

18683 initial records, 18683 valid entries after cleaning.


In [ ]:
data['Balcony'] = data['Balcony'].replace(
    'Առանց պատշգամբ',
    'Առկա չէ'
)

In [ ]:
print(f"Final data shape after cleaning: {data.shape}.")

# Save
data.to_csv("cleaned_data.csv", index=False)

Final data shape after cleaning: (18683, 11).


In [ ]:
data.head()

,New_building,nFloor,Floor,Height,Renovation,Sqm,Rooms,Bathrooms,Balcony,Price,District
0,Ոչ,9,6,2.80,Կապիտալ վերանորոգված,68,2,1,Փակ պատշգամբ,95000.0,Մալաթիա-Սեբաստիա
2,Այո,12,6,3.00,Կապիտալ վերանորոգված,46,2,1,Բաց պատշգամբ,87000.0,Աջափնյակ
3,Ոչ,4,2,3.00,Մասնակի վերանորոգում,50,1,1,Փակ պատշգամբ,130000.0,Կենտրոն
4,Ոչ,5,1,2.75,Կապիտալ վերանորոգված,70,3,1,Մի քանի պատշգամբ,145000.0,Արաբկիր
5,Այո,8,5,3.00,Կապիտալ վերանորոգված,47,2,1,Բաց պատշգամբ,160000.0,Արաբկիր


In [ ]:
data['New_building'].unique()

array(['Ոչ', 'Այո'], dtype=object)

In [ ]:
data['Balcony'].unique()

array(['Փակ պատշգամբ', 'Բաց պատշգամբ', 'Մի քանի պատշգամբ', 'Առկա չէ',
       'Առանց պատշգամբ'], dtype=object)

In [ ]:
data['Renovation'].unique()

array(['Կապիտալ վերանորոգված', 'Մասնակի վերանորոգում', 'Եվրովերանորոգված',
       'Կոսմետիկ վերանորոգում', 'Դիզայներական ոճով վերանորոգված',
       'Չվերանորոգված', 'Հին վերանորոգում'], dtype=object)

In [ ]:
data['District'].unique()

array(['Մալաթիա-Սեբաստիա', 'Աջափնյակ', 'Կենտրոն', 'Արաբկիր', 'Շենգավիթ',
       'Քանաքեռ-Զեյթուն', 'Դավթաշեն', 'Ավան', 'Նորք-Մարաշ', 'Նոր Նորք',
       'Էրեբունի', 'Նուբարաշեն'], dtype=object)

We currently have the following categorical variables:

* New_building -- 'Այո', 'Ոչ'

* Renovation -- 'Դիզայներական ոճով վերանորոգված', 'Հին վերանորոգում',
       'Կոսմետիկ վերանորոգում', 'Կապիտալ վերանորոգված',
       'Եվրովերանորոգված', 'Մասնակի վերանորոգում', 'Չվերանորոգված'

* Balcony -- 'Բաց պատշգամբ', 'Առկա չէ', 'Մի քանի պատշգամբ', 'Փակ պատշգամբ'

* District -- 'Դավթաշեն', 'Էրեբունի', 'Նուբարաշեն', 'Աջափնյակ', 'Արաբկիր',
       'Շենգավիթ', 'Նոր Նորք', 'Կենտրոն', 'Մալաթիա-Սեբաստիա', 'Ավան',
       'Քանաքեռ-Զեյթուն', 'Նորք-Մարաշ'

In [ ]:
def set_categorical_from_data(df, col):
    df[col] = pd.Categorical(
        df[col],
        categories=sorted(df[col].dropna().unique())
    )

In [ ]:
categorical_cols = [
    'New_building',
    'Renovation',
    'Balcony',
    'District'
]

for col in categorical_cols:
    set_categorical_from_data(data, col)

data_dummies = pd.get_dummies(
    data,
    columns=categorical_cols,
    drop_first=True
)

In [ ]:
data_dummies.columns

Index(['nFloor', 'Floor', 'Height', 'Sqm', 'Rooms', 'Bathrooms', 'Price',
       'New_building_Ոչ', 'Renovation_Եվրովերանորոգված',
       'Renovation_Կապիտալ վերանորոգված', 'Renovation_Կոսմետիկ վերանորոգում',
       'Renovation_Հին վերանորոգում', 'Renovation_Մասնակի վերանորոգում',
       'Renovation_Չվերանորոգված', 'Balcony_Առկա չէ', 'Balcony_Բաց պատշգամբ',
       'Balcony_Մի քանի պատշգամբ', 'Balcony_Փակ պատշգամբ', 'District_Ավան',
       'District_Արաբկիր', 'District_Դավթաշեն', 'District_Էրեբունի',
       'District_Կենտրոն', 'District_Մալաթիա-Սեբաստիա', 'District_Նոր Նորք',
       'District_Նորք-Մարաշ', 'District_Նուբարաշեն', 'District_Շենգավիթ',
       'District_Քանաքեռ-Զեյթուն'],
      dtype='object')

In [ ]:
# Check dtypes
X = data_dummies.drop(columns=['Price'])
print(X.dtypes)


nFloor                                int64
Floor                                 int64
Height                              float64
Sqm                                   int64
Rooms                                 int64
Bathrooms                             int64
New_building_Ոչ                        bool
Renovation_Եվրովերանորոգված            bool
Renovation_Կապիտալ վերանորոգված        bool
Renovation_Կոսմետիկ վերանորոգում       bool
Renovation_Հին վերանորոգում            bool
Renovation_Մասնակի վերանորոգում        bool
Renovation_Չվերանորոգված               bool
Balcony_Առկա չէ                        bool
Balcony_Բաց պատշգամբ                   bool
Balcony_Մի քանի պատշգամբ               bool
Balcony_Փակ պատշգամբ                   bool
District_Ավան                          bool
District_Արաբկիր                       bool
District_Դավթաշեն                      bool
District_Էրեբունի                      bool
District_Կենտրոն                       bool
District_Մալաթիա-Սեբաստիա       

In [ ]:
# Convert all boolean columns to integers (0/1)
import numpy as np
X_numeric = X.astype(int)

print("Rank:", np.linalg.matrix_rank(X_numeric.values))
print("Number of regressors:", X_numeric.shape[1])

Rank: 28
Number of regressors: 28


According to the results we got in the previous step, we see that the following claims are sattisfied:

* No perfect multicollinearity

* Each column (regressor) provides unique information.

* There are no columns that are exact linear combinations of others.

In [ ]:
data_dummies.head()

,nFloor,Floor,Height,Sqm,Rooms,Bathrooms,Price,New_building_Ոչ,Renovation_Եվրովերանորոգված,Renovation_Կապիտալ վերանորոգված,...,District_Արաբկիր,District_Դավթաշեն,District_Էրեբունի,District_Կենտրոն,District_Մալաթիա-Սեբաստիա,District_Նոր Նորք,District_Նորք-Մարաշ,District_Նուբարաշեն,District_Շենգավիթ,District_Քանաքեռ-Զեյթուն
0,9,6,2.80,68,2,1,95000.0,True,False,True,...,False,False,False,False,True,False,False,False,False,False
2,12,6,3.00,46,2,1,87000.0,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,4,2,3.00,50,1,1,130000.0,True,False,False,...,False,False,False,True,False,False,False,False,False,False
4,5,1,2.75,70,3,1,145000.0,True,False,True,...,True,False,False,False,False,False,False,False,False,False
5,8,5,3.00,47,2,1,160000.0,False,False,True,...,True,False,False,False,False,False,False,False,False,False


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

X = X_numeric.copy()
X['const'] = 1
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

vif_data = vif_data[vif_data['feature'] != 'const']
vif_data = vif_data.sort_values(by='VIF', ascending=False)
print(vif_data)


                             feature       VIF
14              Balcony_Բաց պատշգամբ  4.109037
21                  District_Կենտրոն  3.493757
16              Balcony_Փակ պատշգամբ  3.443408
18                  District_Արաբկիր  3.005351
3                                Sqm  2.930264
6                    New_building_Ոչ  2.817448
15          Balcony_Մի քանի պատշգամբ  2.816255
8    Renovation_Կապիտալ վերանորոգված  2.735243
13                   Balcony_Առկա չէ  2.358258
4                              Rooms  2.275864
0                             nFloor  2.211402
22         District_Մալաթիա-Սեբաստիա  2.123003
2                             Height  2.107195
7        Renovation_Եվրովերանորոգված  2.051688
10       Renovation_Հին վերանորոգում  2.042409
9   Renovation_Կոսմետիկ վերանորոգում  1.893327
23                 District_Նոր Նորք  1.815301
27          District_Քանաքեռ-Զեյթուն  1.809619
1                              Floor  1.722998
5                          Bathrooms  1.701688
26           

#**Log-Tranformed Price**

We set up the OLS regression with log-transformed Price, which helps:

* Reduce heteroscedasticity

* Make coefficients interpretable as percentage changes

In [ ]:
data_dummies['log_Price'] = np.log(data_dummies['Price'])

In [ ]:
import statsmodels.api as sm

X = X.apply(lambda col: pd.to_numeric(col, errors='coerce') if col.dtype != 'bool'
            else col.astype(int))

# Add the constant (intercept)
X = sm.add_constant(X)
y = pd.to_numeric(data_dummies['log_Price'], errors='coerce')

In [ ]:
import statsmodels.api as sm

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_Price   R-squared:                       0.821
Model:                            OLS   Adj. R-squared:                  0.821
Method:                 Least Squares   F-statistic:                     3063.
Date:                Sat, 27 Dec 2025   Prob (F-statistic):               0.00
Time:                        07:19:14   Log-Likelihood:                -144.92
No. Observations:               18683   AIC:                             347.8
Df Residuals:                   18654   BIC:                             575.1
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
nFloor  

In [ ]:
print("R-squared:", model.rsquared)
print("Adjusted R-squared:", model.rsquared_adj)
print("F-statistic:", model.fvalue)
print("Prob(F-statistic):", model.f_pvalue)

R-squared: 0.8213736135314691
Adjusted R-squared: 0.8211054920121639
F-statistic: 3063.4378607879025
Prob(F-statistic): 0.0


#**Breusch–Pagan Test**

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test = het_breuschpagan(model.resid, model.model.exog)

bp_labels = ['Lagrange multiplier statistic', 'p-value',
             'f-value', 'f p-value']

bp_results = dict(zip(bp_labels, bp_test))
print(bp_results)

{'Lagrange multiplier statistic': np.float64(2987.51710189382), 'p-value': np.float64(0.0), 'f-value': np.float64(126.80887775281877), 'f p-value': np.float64(0.0)}


We reject H₀, meaning heteroscedasticity is present as 'p-value': 4.8e-89

In [ ]:
model_robust = sm.OLS(y, X).fit(cov_type='HC3')
print(model_robust.summary())

                            OLS Regression Results                            
Dep. Variable:              log_Price   R-squared:                       0.821
Model:                            OLS   Adj. R-squared:                  0.821
Method:                 Least Squares   F-statistic:                     2181.
Date:                Fri, 26 Dec 2025   Prob (F-statistic):               0.00
Time:                        23:23:20   Log-Likelihood:                -145.30
No. Observations:               18683   AIC:                             346.6
Df Residuals:                   18655   BIC:                             566.0
Df Model:                          27                                         
Covariance Type:                  HC3                                         
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
nFloor  

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

                             feature         VIF
0                             nFloor    2.211357
1                              Floor    1.722911
2                             Height    2.107195
3                                Sqm    2.930249
4                              Rooms    2.275572
5                          Bathrooms    1.701663
6                    New_building_Ոչ    2.817435
7        Renovation_Եվրովերանորոգված    2.051670
8    Renovation_Կապիտալ վերանորոգված    2.735183
9   Renovation_Կոսմետիկ վերանորոգում    1.893321
10       Renovation_Հին վերանորոգում    2.042408
11   Renovation_Մասնակի վերանորոգում    1.627580
12          Renovation_Չվերանորոգված    1.683885
13              Balcony_Բաց պատշգամբ    2.026345
14          Balcony_Մի քանի պատշգամբ    1.725924
15              Balcony_Փակ պատշգամբ    1.850000
16                     District_Ավան    1.307328
17                  District_Արաբկիր    3.005156
18                 District_Դավթաշեն    1.622743
19                 D